# Shortest paths in navigable graphs

Dijkstra's algorithm over a directed graph given as an adjacency list of the form

```
<source_id> [(neighbor, uncov), (neighbor, uncov), ...]
```

one source per line (the same format produced by `edge_to_coverage_analysis.py` /
`distributed_robust_prune.py`). The `uncov` field is ignored here.

**Edge weights.** The weight of an edge `u -> v` is the out-degree of the
destination vertex: `w(u -> v) = deg(v)`, where `deg(v)` is the number of
out-neighbours of `v`. This depends only on the in-vertex `v`, not on `u`, so
every edge entering `v` costs the same.

In [ ]:
import ast
import heapq
from tqdm import tqdm

In [ ]:
def load_adjacency(path):
    """Load a directed graph from an adjacency-list file.

    Each line is either
        '<source_id> [(neighbor, uncov), ...]'   (source-prefixed), or
        '[(neighbor, uncov), ...]'               (bare; line index is the source).
    The `uncov` field is ignored. Returns adj: dict source -> list of neighbours.
    """
    adj = {}
    with open(path, 'r') as f:
        for i, line in enumerate(tqdm(f, desc='Loading graph')):
            line = line.strip()
            if not line:
                continue
            if line.startswith('['):
                source = i
                neighborhood = ast.literal_eval(line)
            else:
                space = line.index(' ')
                source = int(line[:space])
                neighborhood = ast.literal_eval(line[space + 1:])
            # neighborhood is [(neighbor, uncov), ...]; keep only the neighbour.
            adj[source] = [int(nbr) for (nbr, _uncov) in neighborhood]
    return adj


def out_degrees(adj):
    """deg(v) = number of out-neighbours of v. Vertices that only ever appear as a
    neighbour (never as a source line) have out-degree 0."""
    deg = {u: len(nbrs) for u, nbrs in adj.items()}
    for nbrs in adj.values():
        for v in nbrs:
            deg.setdefault(v, 0)
    return deg

In [ ]:
def dijkstra(adj, deg, source, target=None):
    """Shortest paths from `source` using edge weight w(u -> v) = deg(v).

    Args:
        adj:    dict u -> list of out-neighbours v.
        deg:    dict v -> out-degree of v (the edge weight for any edge into v).
        source: start vertex.
        target: if given, stop early once it is finalised and return
                (distance, path); otherwise return (dist, prev) for all vertices.

    All weights deg(v) >= 0, so Dijkstra is valid. The source has distance 0.
    """
    dist = {source: 0}
    prev = {source: None}
    visited = set()
    pq = [(0, source)]                      # (distance, vertex) min-heap

    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue                        # stale heap entry
        visited.add(u)

        if target is not None and u == target:
            return d, _reconstruct(prev, target)

        for v in adj.get(u, ()):
            if v in visited:
                continue
            nd = d + deg[v]                 # weight of edge u -> v is deg(v)
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))

    if target is not None:
        return float('inf'), None            # target unreachable
    return dist, prev


def _reconstruct(prev, target):
    """Walk `prev` pointers back from target to source, return the forward path."""
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev.get(node)
    path.reverse()
    return path


def shortest_path(adj, deg, source, target):
    """Convenience wrapper: shortest path between two vertices.

    Returns (distance, path). distance is inf and path is None if target is
    unreachable from source.
    """
    return dijkstra(adj, deg, source, target=target)

In [ ]:
# --- Demo on a small hand-built graph ---
# Weight of an edge into v is deg(v): here deg = {0:2, 1:2, 2:1, 3:0}.
demo_lines = {
    0: [(1, 0), (2, 0)],   # 0 -> 1 (w=deg(1)=2),  0 -> 2 (w=deg(2)=1)
    1: [(2, 0), (3, 0)],   # 1 -> 2 (w=1),         1 -> 3 (w=deg(3)=0)
    2: [(3, 0)],           # 2 -> 3 (w=0)
    3: [],
}
demo_adj = {u: [n for (n, _u) in nbrs] for u, nbrs in demo_lines.items()}
demo_deg = out_degrees(demo_adj)
print('degrees:', demo_deg)

for tgt in (3, 2, 1):
    d, path = shortest_path(demo_adj, demo_deg, 0, tgt)
    print(f'0 -> {tgt}: distance={d}, path={path}')
# 0->3: 0->2->3 costs deg(2)+deg(3)=1+0=1; 0->1->3 costs deg(1)+deg(3)=2+0=2.
# Expect distance 1 via [0, 2, 3].

In [ ]:
# --- Run on a real adjacency-list file ---
# ADJ_PATH = '../edge_to_coverage/adj-list-<dataset>.txt'   # point at your file
# adj = load_adjacency(ADJ_PATH)
# deg = out_degrees(adj)
# distance, path = shortest_path(adj, deg, SOURCE, TARGET)
# print('distance:', distance)
# print('path length (edges):', len(path) - 1 if path else None)

## Greedy routing with Euclidean distances

Greedy graph routing toward a target point. Each vertex is a point whose vector
lives in the `'train'` set of an HDF5 file (vertex id = row index), as in
`beam_search/`. From the current node we move to the out-neighbour whose vector
is closest (squared Euclidean) to the target's vector, as long as it strictly
improves on the current node's distance to the target. The walk stops at the
target or at a local minimum (no neighbour closer than the current node).

Unlike Dijkstra, this is a heuristic that follows the graph geometry; it is not
guaranteed to find the target or the shortest path, but it mirrors how a
navigable graph is actually searched.

In [ ]:
import numpy as np
import h5py
from scipy.spatial.distance import cdist


def load_vectors(hdf5_path, group='train'):
    """Load the point vectors from an HDF5 file (as in beam_search/).

    Vertex id is the row index into this array. Returns an (n, d) float array.
    """
    with h5py.File(hdf5_path, 'r') as f:
        return f[group][:]

In [ ]:
def greedy_path(adj, X, source, target, max_steps=None):
    """Greedy graph routing from `source` toward `target` using Euclidean distance.

    Each step moves to the out-neighbour whose vector X[v] is closest (squared
    Euclidean) to the target's vector X[target], provided it strictly improves on
    the current node's distance to the target. Stops when it reaches the target,
    or at a local minimum (no neighbour is closer than the current node).

    Args:
        adj:       dict u -> list of out-neighbours v.
        X:         (n, d) array of point vectors; X[i] is vertex i's vector.
        source:    start vertex id (row index into X).
        target:    goal vertex id (row index into X).
        max_steps: optional cap on hops (default: n, a safe upper bound).

    Returns:
        (reached, path): reached is True iff the walk arrived at target;
        path is the list of visited vertices [source, ..., last].
    """
    if max_steps is None:
        max_steps = X.shape[0]

    tgt_vec = X[target][None, :]                       # (1, d)
    current = source
    path = [current]

    # squared euclidean distance from current node's vector to the target
    cur_dist = float(cdist(X[current][None, :], tgt_vec, metric='sqeuclidean')[0, 0])

    for _ in range(max_steps):
        if current == target:
            return True, path

        nbrs = adj.get(current, [])
        if not nbrs:
            break                                      # dead end

        # distances from each neighbour to the target
        nbr_dists = cdist(X[nbrs], tgt_vec, metric='sqeuclidean').ravel()
        best = int(np.argmin(nbr_dists))
        best_dist = float(nbr_dists[best])

        if best_dist >= cur_dist:
            break                                      # local minimum: no progress

        current = nbrs[best]
        cur_dist = best_dist
        path.append(current)

    return current == target, path

In [ ]:
# --- Demo: greedy routing on a small 1-D point set ---
# Points on a line at x = 0,1,2,3,4; edges chain 0->1->2->3->4 plus a shortcut
# 0->2. Greedy toward target 4 always steps to the neighbour nearest 4.
demo_X = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]])
demo_adj_g = {
    0: [1, 2],   # from 0, neighbours at x=1 and x=2; x=2 is closer to target 4
    1: [2],
    2: [3],
    3: [4],
    4: [],
}
reached, path = greedy_path(demo_adj_g, demo_X, source=0, target=4)
print(f'0 -> 4: reached={reached}, path={path}')   # expect [0, 2, 3, 4] (takes the shortcut)

# A case that gets stuck at a local minimum: target has no in-path from source's region.
stuck_adj = {0: [1], 1: [], 2: [0]}
reached, path = greedy_path(stuck_adj, demo_X, source=0, target=4)
print(f'0 -> 4 (stuck): reached={reached}, path={path}')

In [ ]:
# --- Greedy routing on a real graph + HDF5 vectors ---
# ADJ_PATH  = '../edge_to_coverage/adj-list-<dataset>.txt'   # graph
# HDF5_PATH = '.../<dataset>.hdf5'                           # vectors ('train' set)
# adj = load_adjacency(ADJ_PATH)
# X   = load_vectors(HDF5_PATH)          # X[i] is vertex i's vector
# reached, path = greedy_path(adj, X, SOURCE, TARGET)
# print('reached target:', reached)
# print('hops:', len(path) - 1)